# KNN com dados numéricos e categóricos

Neste notebook montamos um KNN do zero para classificar as amostras em controle (0) ou diabetes (1). Mantemos os números como medidas e comparamos as categorias separadamente. Depois testamos quantos vizinhos consultar e avaliamos o K escolhido nas 104 amostras reservadas.

### Leitura da base

Lemos o arquivo de treinamento padronizado e contamos as amostras e as classes. O arquivo de teste da competição não participa desta etapa.


In [1]:
from pathlib import Path
from collections import Counter
import csv
import random

caminho = Path("dados/treinamento_padronizado.csv")
if not caminho.is_file():
    caminho = Path("../dados/treinamento_padronizado.csv")

with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

classes = Counter(amostra["class"] for amostra in amostras)

print("Total de amostras:", len(amostras))
print("Amostras por classe:", dict(classes))

pacientes = Counter(amostra["PATIENT_CODE"].strip() for amostra in amostras)
if "" in pacientes or any(total > 1 for total in pacientes.values()):
    raise ValueError("Confira os códigos de paciente antes de dividir a base.")

Total de amostras: 520
Amostras por classe: {'1': 270, '0': 250}


### Divisão entre treinamento e validação

Separamos as classes, embaralhamos cada uma com semente 42 e reservamos 20% para validação. Assim, preservamos a mesma divisão usada no notebook anterior: 416 amostras para treinamento e 104 para validação.

As 104 amostras já tiveram um resultado inicial consultado no trabalho anterior; por isso, não são uma avaliação totalmente intocada. O conjunto de teste da competição permanece reservado.


In [2]:
controle = [amostra for amostra in amostras if amostra["class"] == "0"]
diabetes = [amostra for amostra in amostras if amostra["class"] == "1"]

if len(controle) + len(diabetes) != len(amostras):
    raise ValueError("Foi encontrada uma classe diferente de 0 ou 1.")

gerador = random.Random(42)
gerador.shuffle(controle)
gerador.shuffle(diabetes)

quantidade_controle = round(len(controle) * 0.20)
quantidade_diabetes = round(len(diabetes) * 0.20)

validacao = controle[:quantidade_controle] + diabetes[:quantidade_diabetes]
treinamento = controle[quantidade_controle:] + diabetes[quantidade_diabetes:]

print("Treinamento:", len(treinamento), dict(Counter(amostra["class"] for amostra in treinamento)))
print("Validação:", len(validacao), dict(Counter(amostra["class"] for amostra in validacao)))


Treinamento: 416 {'0': 200, '1': 216}
Validação: 104 {'0': 50, '1': 54}


### One-hot encoding dos atributos categóricos

Usamos GENDER, SMOKER, PHYSICAL_ACTIVITY, FAMILY_HISTORY, BLOOD_GROUP e RH_FACTOR. Cada categoria ganha uma coluna de 0 ou 1. Por exemplo, se o grupo sanguíneo for A, a coluna do grupo A recebe 1 e as dos outros grupos recebem 0.

Identificamos as categorias usando as amostras de treinamento. Campos vazios, a categoria desconhecido e o valor N de GENDER recebem __desconhecido__. Se aparecer uma categoria que o treinamento ainda não conhece, usamos __nao_visto__. Com isso, os dois conjuntos podem ser representados pelas mesmas colunas.

In [3]:
colunas_categoricas = [
    "GENDER",
    "SMOKER",
    "PHYSICAL_ACTIVITY",
    "FAMILY_HISTORY",
    "BLOOD_GROUP",
    "RH_FACTOR"
]


def categoria_normalizada(amostra, coluna):
    valor = amostra[coluna].strip()

    if valor == "" or valor == "desconhecido":
        return "__desconhecido__"
    if coluna == "GENDER" and valor == "N":
        return "__desconhecido__"
    return valor


def identificar_categorias(amostras_ajuste):
    categorias_por_coluna = {}

    for coluna in colunas_categoricas:
        categorias_observadas = {
            categoria_normalizada(amostra, coluna)
            for amostra in amostras_ajuste
        }
        categorias_observadas.add("__desconhecido__")
        categorias_observadas.add("__nao_visto__")
        categorias_por_coluna[coluna] = sorted(categorias_observadas)
    return categorias_por_coluna

def codificar_categorias(amostra, categorias):
    resultado = {}

    for coluna in categorias:
        valor = categoria_normalizada(amostra, coluna)

        if valor not in categorias[coluna]:
            valor = "__nao_visto__"

        for categoria in categorias[coluna]:
            nome = f"{coluna}__{categoria}"
            resultado[nome] = 1 if valor == categoria else 0

    return resultado


categorias_por_coluna = identificar_categorias(treinamento)

categoricos_treinamento = [
    codificar_categorias(amostra, categorias_por_coluna)
    for amostra in treinamento
]

categoricos_validacao = [
    codificar_categorias(amostra, categorias_por_coluna)
    for amostra in validacao
]

for coluna, categorias in categorias_por_coluna.items():
    print(f"{coluna}: {categorias}")

print("Amostras codificadas para treinamento:", len(categoricos_treinamento))
print("Amostras codificadas para validação:", len(categoricos_validacao))
print("Colunas geradas:", len(categoricos_treinamento[0]))

GENDER: ['F', 'M', '__desconhecido__', '__nao_visto__']
SMOKER: ['__desconhecido__', '__nao_visto__', 'ex_fumante', 'fumante_atual', 'nunca_fumou']
PHYSICAL_ACTIVITY: ['__desconhecido__', '__nao_visto__', 'intensa', 'leve', 'moderada', 'sedentario']
FAMILY_HISTORY: ['__desconhecido__', '__nao_visto__', 'nao', 'sim']
BLOOD_GROUP: ['A', 'AB', 'B', 'O', '__desconhecido__', '__nao_visto__']
RH_FACTOR: ['+', '-', '__desconhecido__', '__nao_visto__']
Amostras codificadas para treinamento: 416
Amostras codificadas para validação: 104
Colunas geradas: 29


### Preparação dos atributos numéricos

Usamos AGE, HEIGHT, WEIGHT, SYSTOLIC, DIASTOLIC, GLUCOSE, HEMOGLOBIN, TRIGLYCERIDES e SODIUM. Para comparar medidas em unidades diferentes, subtraímos a mediana de cada coluna e dividimos pela diferença entre o terceiro e o primeiro quartil. Essa diferença representa a largura da metade central dos dados.

Calculamos a mediana e os quartis com os valores disponíveis no treinamento. Se um valor estiver ausente, usamos a mediana, que vira zero após o ajuste. Esse zero indica o centro da escala ajustada.

Tratamos como ausentes os valores que não conseguimos ler como números finitos e positivos. Para HEMOGLOBIN, mantemos a premissa de que a coluna representa HbA1c em porcentagem e tratamos valores acima de 100 como ausentes. Os demais valores extremos continuam na base e podem influenciar as distâncias.

In [4]:
import math
from statistics import mean, median, quantiles, stdev

colunas_numericas = [
    "AGE", "HEIGHT", "WEIGHT", "SYSTOLIC", "DIASTOLIC",
    "GLUCOSE", "HEMOGLOBIN", "TRIGLYCERIDES", "SODIUM"
]


def ler_numero(amostra, coluna):
    texto = str(amostra[coluna]).strip().replace(",", ".")

    try:
        valor = float(texto)
    except ValueError:
        return None

    if not math.isfinite(valor) or valor <= 0:
        return None
    if coluna == "HEMOGLOBIN" and valor > 100:
        return None
    return valor


def calcular_parametros_numericos(amostras_ajuste, escala="quartis"):
    if escala not in {"quartis", "desvio"}:
        raise ValueError("A escala deve ser quartis ou desvio.")
    parametros_numericos = {}

    for coluna in colunas_numericas:
        valores = [ler_numero(amostra, coluna) for amostra in amostras_ajuste]
        preenchidos = [valor for valor in valores if valor is not None]

        if len(preenchidos) < 2:
            raise ValueError(f"Valores insuficientes no treinamento para {coluna}.")

        q1, _, q3 = quantiles(preenchidos, n=4, method="inclusive")
        mediana = median(preenchidos)
        if escala == "quartis":
            centro = mediana
            intervalo = q3 - q1
        else:
            centro = mean(preenchidos)
            intervalo = stdev(preenchidos)

        if intervalo == 0:
            raise ValueError(f"Escala igual a zero em {coluna}.")

        parametros_numericos[coluna] = {
            "mediana": mediana,
            "centro": centro,
            "q1": q1,
            "q3": q3,
            "intervalo": intervalo
        }
    return parametros_numericos

def preparar_numericos(amostra, parametros, ignorar_ausentes=()):
    resultado = {}

    for coluna in colunas_numericas:
        valor = ler_numero(amostra, coluna)
        mediana = parametros[coluna]["mediana"]
        centro = parametros[coluna]["centro"]
        intervalo = parametros[coluna]["intervalo"]

        if valor is None and coluna in ignorar_ausentes:
            resultado[coluna] = None
            continue
        if valor is None:
            valor = mediana

        resultado[coluna] = (valor - centro) / intervalo

    return resultado


parametros_numericos = calcular_parametros_numericos(treinamento)

numericos_treinamento = [
    preparar_numericos(amostra, parametros_numericos)
    for amostra in treinamento
]

numericos_validacao = [
    preparar_numericos(amostra, parametros_numericos)
    for amostra in validacao
]

for coluna, parametros in parametros_numericos.items():
    print(
        f"{coluna}: mediana = {parametros['mediana']:.2f}, "
        f"Q1 = {parametros['q1']:.2f}, Q3 = {parametros['q3']:.2f}"
    )

print("Amostras numéricas de treinamento:", len(numericos_treinamento))
print("Amostras numéricas de validação:", len(numericos_validacao))

AGE: mediana = 57.00, Q1 = 48.00, Q3 = 67.25
HEIGHT: mediana = 162.00, Q1 = 157.00, Q3 = 169.00
WEIGHT: mediana = 76.40, Q1 = 63.73, Q3 = 87.18
SYSTOLIC: mediana = 123.00, Q1 = 113.00, Q3 = 133.00
DIASTOLIC: mediana = 81.00, Q1 = 76.00, Q3 = 88.00
GLUCOSE: mediana = 103.00, Q1 = 86.25, Q3 = 131.50
HEMOGLOBIN: mediana = 6.40, Q1 = 5.40, Q3 = 8.70
TRIGLYCERIDES: mediana = 151.00, Q1 = 106.00, Q3 = 204.75
SODIUM: mediana = 140.00, Q1 = 138.00, Q3 = 142.00
Amostras numéricas de treinamento: 416
Amostras numéricas de validação: 104


### Distância numérica e categórica

Nos números, usamos Manhattan. Calculamos a diferença absoluta em cada coluna e tiramos a média dessas diferenças. Se essa média for D, usamos D / (1 + D) para colocá-la na faixa de 0 a 1, sem chegar a 1. Por exemplo, D igual a 1 vira 0,5 e D igual a 3 vira 0,75.

Nas categorias, usamos Jaccard. Comparamos as posições com valor 1 nos dois vetores one-hot. Uma categoria igual aumenta a semelhança. Se o valor estiver desconhecido ou se a categoria não tiver aparecido no treinamento em qualquer uma das amostras, deixamos aquele atributo fora da comparação. Grupo sanguíneo e fator Rh entram como dois atributos separados.

A distância final é a média das duas partes, com peso de 50% para cada uma. Quando nenhuma categoria puder ser comparada, usamos a parte numérica sozinha. Quanto menor a distância final, mais perto ficam as amostras na comparação.

In [5]:
def distancia_numerica(primeira, segunda, pesos=None, metrica="manhattan"):
    if metrica not in {"manhattan", "euclidiana"}:
        raise ValueError("A distância numérica deve ser manhattan ou euclidiana.")
    soma = 0
    soma_pesos = 0

    for coluna in colunas_numericas:
        if primeira[coluna] is None or segunda[coluna] is None:
            continue

        peso = pesos.get(coluna, 1) if pesos is not None else 1
        diferenca = abs(primeira[coluna] - segunda[coluna])
        if metrica == "euclidiana":
            diferenca = diferenca ** 2
        soma += peso * diferenca
        soma_pesos += peso

    media = soma / soma_pesos
    if metrica == "euclidiana":
        media = math.sqrt(media)
    ajustada = media / (1 + media)
    return media, ajustada


def distancia_categorica_jaccard(primeira, segunda, categorias, colunas):
    intersecao = 0
    uniao = 0
    atributos_comparados = 0

    for coluna in colunas:
        nome_desconhecido = f"{coluna}____desconhecido__"
        nome_nao_visto = f"{coluna}____nao_visto__"

        if (
            primeira[nome_desconhecido] == 1
            or segunda[nome_desconhecido] == 1
            or primeira[nome_nao_visto] == 1
            or segunda[nome_nao_visto] == 1
        ):
            continue

        atributos_comparados += 1

        for categoria in categorias[coluna]:
            nome = f"{coluna}__{categoria}"
            valor_1 = primeira[nome]
            valor_2 = segunda[nome]

            if valor_1 == 1 and valor_2 == 1:
                intersecao += 1
            if valor_1 == 1 or valor_2 == 1:
                uniao += 1

    if uniao == 0:
        return None, 0

    distancia = 1 - intersecao / uniao
    return distancia, atributos_comparados


def distancia_mista(
    numerica_1, categorica_1, numerica_2, categorica_2, categorias,
    colunas=None, peso_categorico=0.5, pesos_numericos=None,
    metrica_numerica="manhattan"
):
    media_numerica, numerica_ajustada = distancia_numerica(
        numerica_1, numerica_2, pesos_numericos, metrica_numerica
    )
    if colunas is None:
        colunas = colunas_categoricas

    jaccard, comparados = distancia_categorica_jaccard(
        categorica_1, categorica_2, categorias, colunas
    )

    if jaccard is None:
        final = numerica_ajustada
    else:
        final = (1 - peso_categorico) * numerica_ajustada + peso_categorico * jaccard

    return {
        f"{metrica_numerica}_media": media_numerica,
        "numerica_ajustada": numerica_ajustada,
        "jaccard": jaccard,
        "categoricos_comparados": comparados,
        "distancia_final": final
    }


exemplo_distancia = distancia_mista(
    numericos_treinamento[0], categoricos_treinamento[0],
    numericos_treinamento[1], categoricos_treinamento[1],
    categorias_por_coluna
)

for nome, valor in exemplo_distancia.items():
    print(f"{nome}: {valor}")

manhattan_media: 0.6188604305769895
numerica_ajustada: 0.3822815227847759
jaccard: 0.2857142857142857
categoricos_comparados: 6
distancia_final: 0.3339979042495308


### Divisão para validação cruzada

Para comparar os valores de K, dividimos as amostras de treinamento em cinco partes, preservando aproximadamente a proporção das classes em cada uma. A cada rodada, uma parte será usada para avaliação e as outras quatro para treinamento.

A divisão usa a semente 42 para que possa ser reproduzida. As amostras da validação separada anteriormente não participam dessas cinco partes. Antes de avaliar cada rodada, precisaremos calcular novamente a preparação dos dados usando apenas as quatro partes de treinamento.


In [6]:
partes = [[] for _ in range(5)]
gerador_partes = random.Random(42)

for classe in ["0", "1"]:
    amostras_classe = [amostra for amostra in treinamento if amostra["class"] == classe]
    gerador_partes.shuffle(amostras_classe)

    for indice, amostra in enumerate(amostras_classe):
        partes[indice % 5].append(amostra)

for numero, parte in enumerate(partes, start=1):
    classes = Counter(amostra["class"] for amostra in parte)
    print(f"Parte {numero}: {len(parte)} amostras | Classes: {dict(classes)}")


Parte 1: 84 amostras | Classes: {'0': 40, '1': 44}
Parte 2: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 3: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 4: 83 amostras | Classes: {'0': 40, '1': 43}
Parte 5: 83 amostras | Classes: {'0': 40, '1': 43}


### Como o KNN escolhe a classe

Para classificar uma amostra, calculamos sua distância até cada amostra do treinamento e ordenamos da menor para a maior. Pegamos os K primeiros vizinhos. Cada um dá um voto à sua própria classe. Com K igual a 3, se os vizinhos tiverem classes 1, 0 e 1, a previsão será 1 por dois votos a um.

Usamos K ímpar na comparação, então as duas classes sempre terão contagens diferentes. Se usarmos K par e os votos empatarem, vale a classe do vizinho mais próximo. Quando duas distâncias forem iguais, vale a ordem sorteada com semente 42 nas amostras de referência. A ordenação por distância preserva essa ordem nos empates, inclusive quando dois vizinhos disputam a última vaga.

In [7]:
def montar_referencia(amostras_ajuste, parametros, categorias, ignorar_ausentes=()):
    referencia = []

    for amostra in amostras_ajuste:
        numericos = preparar_numericos(amostra, parametros, ignorar_ausentes)
        categoricos = codificar_categorias(amostra, categorias)
        referencia.append((numericos, categoricos, amostra["class"]))

    random.Random(42).shuffle(referencia)
    return referencia


def ordenar_vizinhos(
    amostra, referencia, parametros, categorias,
    colunas=None, peso_categorico=0.5, pesos_numericos=None,
    ignorar_ausentes=(), metrica_numerica="manhattan"
):
    numericos = preparar_numericos(amostra, parametros, ignorar_ausentes)
    categoricos = codificar_categorias(amostra, categorias)
    vizinhos = []

    for numeros_ref, categorias_ref, classe in referencia:
        distancias = distancia_mista(
            numericos, categoricos, numeros_ref, categorias_ref, categorias,
            colunas, peso_categorico, pesos_numericos, metrica_numerica
        )
        vizinhos.append((distancias["distancia_final"], classe))

    vizinhos.sort(key=lambda vizinho: vizinho[0])
    return vizinhos


def prever_classe(vizinhos, k):
    if not 1 <= k <= len(vizinhos):
        raise ValueError("K deve estar entre 1 e a quantidade de vizinhos.")

    escolhidos = vizinhos[:k]
    votos = Counter(classe for distancia, classe in escolhidos)

    if votos["0"] == votos["1"]:
        return escolhidos[0][1]
    if votos["1"] > votos["0"]:
        return "1"
    return "0"

### Comparação dos valores de K

Testamos K igual a 1, 3, 5, 7, 9 e 11. Na primeira rodada, a parte 1 serve para avaliação e as outras quatro viram a referência. Na segunda rodada, a parte 2 assume esse papel, e assim por diante. Ao final, cada amostra das 416 tem uma previsão para cada K, feita na rodada em que ficou de fora.

A cada rodada, calculamos as medianas, os quartis e as categorias novamente com as quatro partes de referência. Depois ordenamos os vizinhos uma vez para cada amostra de avaliação. Essa mesma ordem serve para todos os valores de K. Por exemplo, K igual a 3 usa os três primeiros e K igual a 5 inclui também o quarto e o quinto.

In [8]:
valores_k = [1, 3, 5, 7, 9, 11]
previsoes_por_rodada = []

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = identificar_categorias(amostras_ajuste)
    referencia = montar_referencia(amostras_ajuste, parametros_rodada, categorias_rodada)

    previstas_por_k = {k: [] for k in valores_k}
    reais = []

    for amostra in amostras_avaliacao:
        vizinhos = ordenar_vizinhos(
            amostra, referencia, parametros_rodada, categorias_rodada
        )
        for k in valores_k:
            previstas_por_k[k].append(prever_classe(vizinhos, k))
        reais.append(amostra["class"])

    previsoes_por_rodada.append((reais, previstas_por_k))
    print(
        f"Rodada {indice + 1} | "
        f"Referência: {len(amostras_ajuste)} | Avaliação: {len(reais)}"
    )

Rodada 1 | Referência: 332 | Avaliação: 84
Rodada 2 | Referência: 333 | Avaliação: 83
Rodada 3 | Referência: 333 | Avaliação: 83
Rodada 4 | Referência: 333 | Avaliação: 83
Rodada 5 | Referência: 333 | Avaliação: 83


#### Como lemos as métricas

- **Acurácia** é a proporção de amostras que acertamos.
- **Precisão** responde, para cada classe, quantas amostras que receberam aquela previsão realmente pertenciam a ela.
- **Recall** responde, para cada classe, quantas amostras que realmente pertenciam a ela foram encontradas pelo modelo.
- **F1 macro** calcula o F1 de cada classe, combinando precisão e recall, e tira a média dos dois. Assim, controle e diabetes têm o mesmo peso no resultado.

Na matriz de confusão, as linhas representam as classes reais e as colunas representam as previsões. Usamos a ordem 0, 1 nos dois sentidos. Os acertos ficam na diagonal.

In [9]:
def calcular_metricas(reais, previstas):
    pares = Counter(zip(reais, previstas))
    resultado = {
        "acuracia": sum(pares[classe, classe] for classe in ["0", "1"]) / len(reais)
    }
    valores_f1 = []

    for classe in ["0", "1"]:
        acertos = pares[classe, classe]
        total_real = sum(pares[classe, prevista] for prevista in ["0", "1"])
        total_previsto = sum(pares[real, classe] for real in ["0", "1"])

        precisao = acertos / total_previsto if total_previsto else 0
        recall = acertos / total_real if total_real else 0
        f1 = 2 * precisao * recall / (precisao + recall) if precisao + recall else 0

        resultado[f"precisao_{classe}"] = precisao
        resultado[f"recall_{classe}"] = recall
        valores_f1.append(f1)

    resultado["precisao_macro"] = (resultado["precisao_0"] + resultado["precisao_1"]) / 2
    resultado["f1_macro"] = sum(valores_f1) / 2
    resultado["matriz"] = [[pares[real, prevista] for prevista in ["0", "1"]]
                           for real in ["0", "1"]]
    return resultado

#### Escolha de K

Para cada K, calculamos as métricas em cada rodada e tiramos a média das cinco. O desvio padrão do F1 macro mostra quanto o resultado variou entre as partes.

Escolhemos o K pela maior média de F1 macro e, se duas médias forem exatamente iguais, ficamos com o menor K. Também olhamos a acurácia para comparar com a meta de 90% e os dois recalls para ver como o modelo se sai em cada classe.

In [10]:
from statistics import mean, stdev

resultados_por_k = {k: [] for k in valores_k}

for reais, previstas_por_k in previsoes_por_rodada:
    for k in valores_k:
        metricas = calcular_metricas(reais, previstas_por_k[k])
        resultados_por_k[k].append(metricas)

nomes_metricas = [
    "acuracia", "f1_macro", "precisao_macro", "precisao_0", "precisao_1",
    "recall_0", "recall_1"
]
medias_por_k = {}

print(" K | Acurácia | F1 macro | Desvio F1 | Precisão macro | Recall 0 | Recall 1")
for k, rodadas in resultados_por_k.items():
    medias = {}
    for nome in nomes_metricas:
        medias[nome] = mean(rodada[nome] for rodada in rodadas)
    medias_por_k[k] = medias
    desvio_f1 = stdev(rodada["f1_macro"] for rodada in rodadas)

    print(
        f"{k:>2} | {medias['acuracia']:>8.2%} | {medias['f1_macro']:>8.2%} | "
        f"{desvio_f1:>9.2%} | {medias['precisao_macro']:>14.2%} | "
        f"{medias['recall_0']:>8.2%} | {medias['recall_1']:>8.2%}"
    )

melhor_k = min(valores_k, key=lambda k: (-medias_por_k[k]["f1_macro"], k))
print(f"\nK escolhido: {melhor_k}")

 K | Acurácia | F1 macro | Desvio F1 | Precisão macro | Recall 0 | Recall 1
 1 |   67.05% |   66.97% |     6.98% |         67.17% |   68.50% |   65.69%
 3 |   65.13% |   65.01% |     4.32% |         65.16% |   63.00% |   67.09%
 5 |   67.05% |   66.91% |     3.81% |         67.33% |   67.50% |   66.60%
 7 |   65.62% |   65.48% |     2.63% |         65.82% |   66.50% |   64.78%
 9 |   67.79% |   67.66% |     4.43% |         68.11% |   71.00% |   64.81%
11 |   65.61% |   65.47% |     3.76% |         66.00% |   68.50% |   62.95%

K escolhido: 9


### Avaliação nas amostras reservadas

Agora usamos as 416 amostras de treinamento como referência e aplicamos o K escolhido às 104 amostras que separamos no início. Para isso, usamos as medianas, os quartis e as categorias calculados com as 416 amostras.

Os resultados abaixo descrevem esta divisão. Como essas 104 amostras já foram consultadas em uma versão anterior do projeto, a avaliação externa continua sendo uma etapa separada.

In [11]:
referencia_final = montar_referencia(treinamento, parametros_numericos, categorias_por_coluna)
reais_validacao = []
previstas_validacao = []

for amostra in validacao:
    vizinhos = ordenar_vizinhos(
        amostra, referencia_final, parametros_numericos, categorias_por_coluna
    )
    reais_validacao.append(amostra["class"])
    previstas_validacao.append(prever_classe(vizinhos, melhor_k))

metricas_validacao = calcular_metricas(reais_validacao, previstas_validacao)
matriz = metricas_validacao["matriz"]
acertos = matriz[0][0] + matriz[1][1]

print(f"K = {melhor_k} | Acertos: {acertos} de {len(validacao)}")
print(f"Acurácia: {metricas_validacao['acuracia']:.2%}")
print(f"F1 macro: {metricas_validacao['f1_macro']:.2%}")
print(f"Precisão macro: {metricas_validacao['precisao_macro']:.2%}")

for indice, classe in enumerate(["0", "1"]):
    print(
        f"Classe {classe} | Precisão: {metricas_validacao[f'precisao_{classe}']:.2%} | "
        f"Recall: {metricas_validacao[f'recall_{classe}']:.2%} | "
        f"Total real: {sum(matriz[indice])}"
    )

print("Matriz de confusão (linhas reais, colunas previstas):")
print("            Prev. 0  Prev. 1")
print(f"Real 0      {matriz[0][0]:>7}  {matriz[0][1]:>7}")
print(f"Real 1      {matriz[1][0]:>7}  {matriz[1][1]:>7}")

K = 9 | Acertos: 71 de 104
Acurácia: 68.27%
F1 macro: 68.20%
Precisão macro: 68.22%
Classe 0 | Precisão: 67.35% | Recall: 66.00% | Total real: 50
Classe 1 | Precisão: 69.09% | Recall: 70.37% | Total real: 54
Matriz de confusão (linhas reais, colunas previstas):
            Prev. 0  Prev. 1
Real 0           33       17
Real 1           16       38


#### Leitura do resultado

Nesta execução, K igual a 9 teve o maior F1 macro médio na validação cruzada, com 67,66%. A diferença para K igual a 1 ou 5 foi menor que um ponto percentual, então o resultado aponta uma escolha para este experimento.

Nas 104 amostras reservadas, acertamos 71. Das 50 amostras de controle, 17 foram classificadas como diabetes. Das 54 amostras de diabetes, 16 foram classificadas como controle. A acurácia ficou em 68,27%, abaixo da meta de 90%.


### O que muda quando usamos um atributo categórico por vez?

Começamos com os nove atributos numéricos e usamos Manhattan para ordenar os vizinhos. Depois repetimos a comparação três vezes. Em cada versão, acrescentamos apenas um atributo ao modelo numérico: histórico familiar, tabagismo ou atividade física.

Nas três versões, damos 90% do peso aos números e 10% à categoria. É um ponto de partida para ver se a categoria ajuda quando tem pouca influência na escolha dos vizinhos. Com uma única categoria, Jaccard vale 0 se os dois valores são iguais e 1 se são diferentes. Se o valor estiver indisponível em qualquer uma das duas amostras, usamos só os números naquela comparação.

Mantemos K igual a 9, a votação simples e as mesmas cinco divisões. A preparação continua sendo refeita em cada rodada com as quatro partes de referência. Dessa vez, comparamos os resultados apenas nas cinco rodadas.

In [12]:
k_comparacao = 9
versoes = {
    "Só numéricos": [],
    "+ Histórico familiar": ["FAMILY_HISTORY"],
    "+ Tabagismo": ["SMOKER"],
    "+ Atividade física": ["PHYSICAL_ACTIVITY"],
}
resultados_atributos = {nome: [] for nome in versoes}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = identificar_categorias(amostras_ajuste)
    referencia = montar_referencia(amostras_ajuste, parametros_rodada, categorias_rodada)
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for nome, colunas in versoes.items():
        previstas = []

        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia, parametros_rodada, categorias_rodada,
                colunas=colunas, peso_categorico=0.10
            )
            previstas.append(prever_classe(vizinhos, k_comparacao))

        resultados_atributos[nome].append(calcular_metricas(reais, previstas))

#### Comparação com o modelo numérico

Lemos cada linha em relação a “Só numéricos”. A diferença de F1 mostra quantos pontos percentuais a versão ganhou ou perdeu na média. Também contamos em quantas das cinco rodadas a versão teve F1 maior, porque uma melhora média pode vir de uma única rodada.

In [13]:
base_numerica = resultados_atributos["Só numéricos"]
f1_base = mean(rodada["f1_macro"] for rodada in base_numerica)

print("Versão                 | Acurácia | F1 macro | Dif. F1 (p.p.) | Recall 0 | Recall 1 | Rodadas melhores")
for nome, rodadas in resultados_atributos.items():
    acuracia = mean(rodada["acuracia"] for rodada in rodadas)
    f1 = mean(rodada["f1_macro"] for rodada in rodadas)
    recall_0 = mean(rodada["recall_0"] for rodada in rodadas)
    recall_1 = mean(rodada["recall_1"] for rodada in rodadas)

    diferenca = f"{100 * (f1 - f1_base):+.2f}" if nome != "Só numéricos" else "-"
    vitorias = sum(
        rodada["f1_macro"] > base["f1_macro"]
        for rodada, base in zip(rodadas, base_numerica)
    )
    rodadas_melhores = f"{vitorias}/5" if nome != "Só numéricos" else "-"

    print(
        f"{nome:22} | {acuracia:>8.2%} | {f1:>8.2%} | {diferenca:>14} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%} | {rodadas_melhores:>16}"
    )

Versão                 | Acurácia | F1 macro | Dif. F1 (p.p.) | Recall 0 | Recall 1 | Rodadas melhores
Só numéricos           |   83.40% |   83.35% |              - |   90.00% |   77.28% |                -
+ Histórico familiar   |   83.64% |   83.60% |          +0.25 |   89.50% |   78.19% |              2/5
+ Tabagismo            |   83.87% |   83.86% |          +0.51 |   88.50% |   79.58% |              4/5
+ Atividade física     |   82.19% |   82.11% |          -1.25 |   91.00% |   74.02% |              1/5


#### O que encontramos

O tabagismo teve a maior média de F1 macro e melhorou o F1 em quatro das cinco rodadas. Também aumentou o recall de diabetes de 77,28% para 79,58%, enquanto o recall de controle caiu de 90,00% para 88,50%.

Histórico familiar teve um ganho médio menor e melhorou em duas rodadas. Atividade física reduziu o F1 médio nesta configuração. Como os ganhos foram pequenos e vieram das mesmas cinco divisões usadas na comparação, tratamos tabagismo como candidato para uma avaliação posterior.


### Quantos vizinhos usar e quanto devem pesar os exames?

Agora comparamos K igual a 1, 3, 5, 7, 9 e 11 com os nove atributos numéricos e tabagismo. O tabagismo continua com 10% do peso quando há um valor conhecido para as duas amostras comparadas. Mantemos a votação simples, o preenchimento pela mediana e as mesmas cinco divisões.

Testamos duas formas de calcular a parte numérica. Na primeira, todas as diferenças têm peso 1. Na segunda, a diferença de glicose e a de hemoglobina têm peso 2, e as outras sete mantêm peso 1. Somamos as diferenças com seus pesos e dividimos pela soma dos pesos. Por exemplo, uma diferença ajustada de 0,4 na glicose contribui com 0,8 para a soma na segunda versão. A regra de converter a média D em D / (1 + D) continua a mesma.

Comparamos os seis valores de K nos dois cenários. Olhar os dois lados da tabela para o mesmo K mostra o efeito de dar mais peso aos exames.

In [14]:
pesos_exames = {
    "Normal": {},
    "Dobrado": {"GLUCOSE": 2, "HEMOGLOBIN": 2},
}
resultados_pesos = {
    (nome, k): [] for nome in pesos_exames for k in valores_k
}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = identificar_categorias(amostras_ajuste)
    referencia = montar_referencia(amostras_ajuste, parametros_rodada, categorias_rodada)
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for nome, pesos in pesos_exames.items():
        previstas_por_k = {k: [] for k in valores_k}

        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia, parametros_rodada, categorias_rodada,
                colunas=["SMOKER"], peso_categorico=0.10, pesos_numericos=pesos
            )
            for k in valores_k:
                previstas_por_k[k].append(prever_classe(vizinhos, k))

        for k, previstas in previstas_por_k.items():
            resultados_pesos[nome, k].append(calcular_metricas(reais, previstas))

#### Comparação dos resultados

A tabela traz as médias das cinco rodadas. A acurácia mostra o total de acertos, e os recalls ajudam a ver em qual classe os acertos mudaram. Ao final, mostramos as versões com maior acurácia e maior F1 macro, porque as duas métricas podem apontar escolhas diferentes.

In [15]:
medias_pesos = {}

print("Peso dos exames |  K | Acurácia | F1 macro | Recall 0 | Recall 1")
for (nome, k), rodadas in resultados_pesos.items():
    medias = {}
    for metrica in ["acuracia", "f1_macro", "recall_0", "recall_1"]:
        medias[metrica] = mean(rodada[metrica] for rodada in rodadas)
    medias_pesos[nome, k] = medias

    print(
        f"{nome:15} | {k:>2} | {medias['acuracia']:>8.2%} | "
        f"{medias['f1_macro']:>8.2%} | {medias['recall_0']:>8.2%} | "
        f"{medias['recall_1']:>8.2%}"
    )

for metrica, nome_metrica in [("acuracia", "acurácia"), ("f1_macro", "F1 macro")]:
    maior_media = max(medias[metrica] for medias in medias_pesos.values())
    print()
    for (nome, k), medias in medias_pesos.items():
        if medias[metrica] == maior_media:
            print(f"Maior {nome_metrica}: {maior_media:.2%} | Peso {nome.lower()} | K = {k}")

Peso dos exames |  K | Acurácia | F1 macro | Recall 0 | Recall 1
Normal          |  1 |   79.33% |   79.31% |   81.00% |   77.78%
Normal          |  3 |   83.17% |   83.15% |   89.00% |   77.76%
Normal          |  5 |   81.97% |   81.95% |   85.50% |   78.70%
Normal          |  7 |   84.36% |   84.35% |   88.50% |   80.53%
Normal          |  9 |   83.87% |   83.86% |   88.50% |   79.58%
Normal          | 11 |   84.36% |   84.33% |   90.00% |   79.11%
Dobrado         |  1 |   81.24% |   81.23% |   84.50% |   78.25%
Dobrado         |  3 |   84.84% |   84.83% |   89.00% |   81.00%
Dobrado         |  5 |   84.85% |   84.84% |   89.50% |   80.55%
Dobrado         |  7 |   85.09% |   85.08% |   90.50% |   80.10%
Dobrado         |  9 |   86.05% |   86.03% |   92.50% |   80.06%
Dobrado         | 11 |   88.21% |   88.18% |   93.50% |   83.28%

Maior acurácia: 88.21% | Peso dobrado | K = 11

Maior F1 macro: 88.18% | Peso dobrado | K = 11


#### O que encontramos

Com o peso dobrado para glicose e hemoglobina, a acurácia média subiu em todos os valores de K testados. K igual a 11 teve a maior acurácia (88,21%) e o maior F1 macro (88,18%).

Mantendo K igual a 11, o peso dobrado aumentou o recall de controle de 90,00% para 93,50% e o de diabetes de 79,11% para 83,28%. A acurácia ficou 1,79 ponto percentual abaixo da meta de 90%. Esses resultados são das cinco rodadas usadas para comparar as configurações.


### Avaliação da configuração escolhida

Fixamos K igual a 11, peso dobrado para glicose e hemoglobina e tabagismo com 10% da distância quando podemos compará-lo. Antes de preparar a submissão, aplicamos essa versão às 104 amostras reservadas, usando como referência as 416 do treinamento. Essas 104 amostras já foram usadas em avaliações anteriores, então este resultado serve como referência local.

In [16]:
k_submissao = 11
pesos_submissao = {"GLUCOSE": 2, "HEMOGLOBIN": 2}

def prever_para_submissao(amostras_avaliacao, referencia, parametros, categorias):
    previsoes = []

    for amostra in amostras_avaliacao:
        vizinhos = ordenar_vizinhos(
            amostra, referencia, parametros, categorias,
            colunas=["SMOKER"], peso_categorico=0.10, pesos_numericos=pesos_submissao
        )
        previsoes.append(prever_classe(vizinhos, k_submissao))

    return previsoes

previsoes_reservadas = prever_para_submissao(
    validacao, referencia_final, parametros_numericos, categorias_por_coluna
)
metricas_reservadas = calcular_metricas(reais_validacao, previsoes_reservadas)
matriz_reservada = metricas_reservadas["matriz"]
acertos_reservados = matriz_reservada[0][0] + matriz_reservada[1][1]

print(f"Acertos: {acertos_reservados} de {len(validacao)}")
print(f"Acurácia: {metricas_reservadas['acuracia']:.2%}")
print(f"F1 macro: {metricas_reservadas['f1_macro']:.2%}")
print(f"Precisão macro: {metricas_reservadas['precisao_macro']:.2%}")
print(f"Recall de controle: {metricas_reservadas['recall_0']:.2%}")
print(f"Recall de diabetes: {metricas_reservadas['recall_1']:.2%}")

Acertos: 90 de 104
Acurácia: 86.54%
F1 macro: 86.53%
Precisão macro: 86.54%
Recall de controle: 88.00%
Recall de diabetes: 85.19%


### Previsões para o Kaggle

Para a submissão, usamos as 520 amostras rotuladas da base padronizada como referência. Calculamos de novo a mediana e os quartis com essas 520 amostras. O conjunto de teste entra apenas para receber as previsões.

O arquivo de teste ainda está no formato original. Aplicamos nele as regras da exploração para os campos que este KNN usa. A altura em metros vira centímetros, as idades 0, -1 e 999 viram campos vazios, a pressão é separada em sistólica e diastólica e os nomes de tabagismo são traduzidos para as mesmas categorias do treinamento. As demais medidas passam pela mesma leitura numérica já usada pelo modelo.

In [17]:
def padronizar_entrada_teste(amostra):
    entrada = amostra.copy()

    if entrada["AGE"].strip() in {"0", "-1", "999"}:
        entrada["AGE"] = ""

    altura = ler_numero(entrada, "HEIGHT")
    if altura is not None and altura < 3:
        entrada["HEIGHT"] = altura * 100

    pressao = entrada["BLOOD_PRESSURE"].strip()
    if pressao in {"", "?"}:
        entrada["SYSTOLIC"] = ""
        entrada["DIASTOLIC"] = ""
    else:
        pressao = pressao.replace("x", "/").replace(" ", "")
        sistolica, diastolica = pressao.split("/")
        entrada["SYSTOLIC"] = int(sistolica)
        entrada["DIASTOLIC"] = int(diastolica)

    mapa_tabagismo = {
        "never": "nunca_fumou",
        "former": "ex_fumante",
        "ex-smoker": "ex_fumante",
        "current": "fumante_atual",
        "yes": "fumante_atual",
        "": "",
    }
    tabagismo = entrada["SMOKER"].strip().lower()
    entrada["SMOKER"] = mapa_tabagismo.get(tabagismo, tabagismo)
    return entrada

caminho_teste = caminho.parent / "kaggle-test" / "test.csv"

with caminho_teste.open(encoding="utf-8-sig", newline="") as arquivo_teste:
    amostras_teste = list(csv.DictReader(arquivo_teste))

ids_teste = [amostra["id"] for amostra in amostras_teste]
if not ids_teste or any(not id_amostra.strip() for id_amostra in ids_teste):
    raise ValueError("O teste precisa ter amostras com IDs preenchidos.")
if len(set(ids_teste)) != len(ids_teste):
    raise ValueError("Foram encontrados IDs repetidos no teste.")

entradas_teste = [padronizar_entrada_teste(amostra) for amostra in amostras_teste]
parametros_submissao = calcular_parametros_numericos(amostras)
categorias_submissao = {"SMOKER": identificar_categorias(amostras)["SMOKER"]}
referencia_submissao = montar_referencia(amostras, parametros_submissao, categorias_submissao)
previsoes_teste = prever_para_submissao(
    entradas_teste, referencia_submissao, parametros_submissao, categorias_submissao
)
print(f"Amostras de referência: {len(referencia_submissao)}")
print(f"Previsões para o teste: {len(previsoes_teste)}")

Amostras de referência: 520
Previsões para o teste: 312


#### Arquivo de submissão

O Kaggle pede as colunas id e class. Usamos o id original de cada linha do teste e a classe prevista, 0 ou 1. Mantemos a ordem do arquivo de teste e salvamos o resultado na pasta resultados.

In [18]:
caminho_submissao = caminho.parent.parent / "resultados" / "submissao_knn_k11.csv"
linhas_submissao = list(zip(ids_teste, previsoes_teste, strict=True))

if any(classe not in {"0", "1"} for id_amostra, classe in linhas_submissao):
    raise ValueError("As previsões devem pertencer às classes 0 ou 1.")

with caminho_submissao.open("w", encoding="utf-8", newline="") as arquivo_submissao:
    escritor = csv.writer(arquivo_submissao)
    escritor.writerow(["id", "class"])
    escritor.writerows(linhas_submissao)

print(f"Arquivo salvo: resultados/{caminho_submissao.name}")
print(f"Linhas de previsão: {len(linhas_submissao)}")

Arquivo salvo: resultados/submissao_knn_k11.csv
Linhas de previsão: 312


### Como lidar com glicose e hemoglobina ausentes?

A primeira submissão marcou 0,82897 no placar público do Kaggle. Antes de mudar o modelo, conferimos os erros nas 104 amostras reservadas para ver se eles se concentravam em amostras sem os dois exames ou sem um deles.

Depois comparamos duas formas de lidar com um valor ausente. Na primeira, preenchemos o exame com a mediana, como antes. Na segunda, deixamos o exame fora da distância quando falta em pelo menos uma das duas amostras. Nesse caso, o peso do exame também sai da soma dos pesos usada para calcular a média. Testamos essa regra para cada exame separadamente e para os dois juntos.

Mantivemos K igual a 11, os pesos dos exames, tabagismo e as mesmas cinco divisões. As outras medidas continuam sendo preenchidas pela mediana quando faltam.

In [19]:
erros_por_grupo = {}

for amostra, prevista in zip(validacao, previsoes_reservadas):
    glicose_ausente = ler_numero(amostra, "GLUCOSE") is None
    hemoglobina_ausente = ler_numero(amostra, "HEMOGLOBIN") is None

    if glicose_ausente and hemoglobina_ausente:
        grupo = "Faltam os dois"
    elif glicose_ausente:
        grupo = "Falta só glicose"
    elif hemoglobina_ausente:
        grupo = "Falta só hemoglobina"
    else:
        grupo = "Dois exames disponíveis"

    totais = erros_por_grupo.setdefault(grupo, {"amostras": 0, "erros": 0})
    totais["amostras"] += 1
    if prevista != amostra["class"]:
        totais["erros"] += 1

for grupo, totais in erros_por_grupo.items():
    print(f"{grupo}: {totais['erros']} erros em {totais['amostras']} amostras")

Dois exames disponíveis: 9 erros em 74 amostras
Falta só hemoglobina: 2 erros em 21 amostras
Faltam os dois: 2 erros em 5 amostras
Falta só glicose: 1 erros em 4 amostras


#### Comparação nas cinco rodadas

Cada linha mostra quais exames ficam fora da distância quando faltam. A linha da mediana corresponde à configuração que enviamos. As médias ajudam a comparar as versões, e a última coluna mostra quantas vezes cada versão teve F1 macro maior que a versão enviada.

In [20]:
regras_exames = {
    "Mediana para os dois": (),
    "Ignorar glicose ausente": ("GLUCOSE",),
    "Ignorar hemoglobina ausente": ("HEMOGLOBIN",),
    "Ignorar ambos se ausentes": ("GLUCOSE", "HEMOGLOBIN"),
}
resultados_ausentes = {nome: [] for nome in regras_exames}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for nome, ignorar in regras_exames.items():
        referencia = montar_referencia(
            amostras_ajuste, parametros_rodada, categorias_rodada, ignorar
        )
        previstas = []
        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia, parametros_rodada, categorias_rodada,
                colunas=["SMOKER"], peso_categorico=0.10,
                pesos_numericos=pesos_submissao, ignorar_ausentes=ignorar
            )
            previstas.append(prever_classe(vizinhos, k_submissao))

        resultados_ausentes[nome].append(calcular_metricas(reais, previstas))

mediana_exames = resultados_ausentes["Mediana para os dois"]
print("Regra                      | Acurácia | F1 macro | Recall 0 | Recall 1 | Rodadas melhores")
for nome, rodadas in resultados_ausentes.items():
    acuracia = mean(r["acuracia"] for r in rodadas)
    f1 = mean(r["f1_macro"] for r in rodadas)
    recall_0 = mean(r["recall_0"] for r in rodadas)
    recall_1 = mean(r["recall_1"] for r in rodadas)
    melhores = sum(r["f1_macro"] > b["f1_macro"] for r, b in zip(rodadas, mediana_exames))
    placar = "-" if nome == "Mediana para os dois" else f"{melhores}/5"

    print(
        f"{nome:26} | {acuracia:>8.2%} | {f1:>8.2%} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%} | {placar:>16}"
    )

Regra                      | Acurácia | F1 macro | Recall 0 | Recall 1 | Rodadas melhores
Mediana para os dois       |   88.21% |   88.18% |   93.50% |   83.28% |                -
Ignorar glicose ausente    |   88.69% |   88.68% |   92.50% |   85.15% |              2/5
Ignorar hemoglobina ausente |   85.33% |   85.32% |   91.00% |   80.07% |              1/5
Ignorar ambos se ausentes  |   85.81% |   85.79% |   91.00% |   80.98% |              0/5


#### O que encontramos

Nas amostras reservadas, foram 9 erros em 74 casos com os dois exames disponíveis e 5 em 30 casos com algum exame ausente. O grupo sem os dois exames tinha só cinco amostras, então é difícil concluir muito a partir dessa divisão.

Nas cinco rodadas, deixar de fora apenas a glicose ausente melhorou o F1 macro médio de 88,18% para 88,68%, com melhora em duas rodadas, empate em uma e piora em duas. O recall de diabetes aumentou, enquanto o de controle caiu. Deixar a hemoglobina ausente fora da distância piorou o F1 médio, com ou sem glicose.

O ganho médio da regra para glicose foi pequeno e variou entre as rodadas. A versão enviada ao Kaggle continua sendo a referência deste experimento.

### Teste com mais vizinhos

K igual a 11 foi o maior valor da comparação anterior. Agora mantemos 11 como referência e testamos 13, 15, 17 e 21. Mantemos a votação simples, o peso dobrado para glicose e hemoglobina e 10% para tabagismo quando podemos compará-lo.

Repetimos o teste com as duas regras para glicose ausente. Uma usa a mediana, como na versão enviada ao Kaggle. A outra deixa a glicose fora da distância quando falta em uma das amostras. Para cada regra, ordenamos os vizinhos uma vez por amostra. K igual a 13 usa os mesmos 11 primeiros e inclui mais dois votos.

Comparamos as médias das mesmas cinco rodadas. Para cada linha, também contamos em quantas rodadas o F1 melhorou em relação a K igual a 11 com a mesma regra para glicose.

In [21]:
valores_k_maiores = [11, 13, 15, 17, 21]
regras_glicose = {
    "Preencher pela mediana": (),
    "Ignorar glicose ausente": ("GLUCOSE",),
}
resultados_k_maiores = {
    (regra, k): [] for regra in regras_glicose for k in valores_k_maiores
}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for regra, ignorar in regras_glicose.items():
        referencia = montar_referencia(
            amostras_ajuste, parametros_rodada, categorias_rodada, ignorar
        )
        previstas_por_k = {k: [] for k in valores_k_maiores}

        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia, parametros_rodada, categorias_rodada,
                colunas=["SMOKER"], peso_categorico=0.10,
                pesos_numericos=pesos_submissao, ignorar_ausentes=ignorar
            )
            for k in valores_k_maiores:
                previstas_por_k[k].append(prever_classe(vizinhos, k))

        for k, previstas in previstas_por_k.items():
            resultados_k_maiores[regra, k].append(calcular_metricas(reais, previstas))

In [22]:
medias_k_maiores = {}
print("Regra para glicose      |  K | Acurácia | F1 macro | Recall 0 | Recall 1 | Rodadas melhores")

for (regra, k), rodadas in resultados_k_maiores.items():
    medias = {}
    for metrica in ["acuracia", "f1_macro", "recall_0", "recall_1"]:
        medias[metrica] = mean(rodada[metrica] for rodada in rodadas)
    medias_k_maiores[regra, k] = medias

    base = resultados_k_maiores[regra, 11]
    melhores = sum(
        rodada["f1_macro"] > base_rodada["f1_macro"]
        for rodada, base_rodada in zip(rodadas, base)
    )
    placar = "-" if k == 11 else f"{melhores}/5"

    print(
        f"{regra:23} | {k:>2} | {medias['acuracia']:>8.2%} | "
        f"{medias['f1_macro']:>8.2%} | {medias['recall_0']:>8.2%} | "
        f"{medias['recall_1']:>8.2%} | {placar:>16}"
    )

Regra para glicose      |  K | Acurácia | F1 macro | Recall 0 | Recall 1 | Rodadas melhores
Preencher pela mediana  | 11 |   88.21% |   88.18% |   93.50% |   83.28% |                -
Preencher pela mediana  | 13 |   87.49% |   87.46% |   93.50% |   81.90% |              0/5
Preencher pela mediana  | 15 |   87.25% |   87.23% |   94.00% |   80.99% |              1/5
Preencher pela mediana  | 17 |   86.78% |   86.76% |   93.00% |   81.00% |              3/5
Preencher pela mediana  | 21 |   86.29% |   86.25% |   94.50% |   78.68% |              0/5
Ignorar glicose ausente | 11 |   88.69% |   88.68% |   92.50% |   85.15% |                -
Ignorar glicose ausente | 13 |   87.73% |   87.72% |   92.50% |   83.30% |              0/5
Ignorar glicose ausente | 15 |   86.77% |   86.74% |   93.00% |   80.98% |              1/5
Ignorar glicose ausente | 17 |   87.25% |   87.23% |   93.50% |   81.44% |              1/5
Ignorar glicose ausente | 21 |   86.53% |   86.50% |   93.50% |   80.06% |      

#### O que encontramos

K igual a 11 teve a maior acurácia e o maior F1 macro médio com as duas regras para glicose. Ao passar de 11 para 21 vizinhos na versão que ignora a glicose ausente, o recall de diabetes caiu de 85,15% para 80,06%, enquanto o de controle passou de 92,50% para 93,50%.

Alguns valores de K acima de 11 melhoraram em rodadas isoladas. Na média, ficamos com K igual a 11 entre os valores comparados.


### Pesos dos exames, escolha dos atributos e votação

Usamos como referência a versão com K igual a 11, peso 2 para glicose e hemoglobina e tabagismo com 10% da distância quando pode ser comparado. Nessa versão, a glicose fica fora da distância quando está ausente e a hemoglobina ausente recebe a mediana.

Mudamos um ponto por vez para comparar cada alternativa com a referência. Primeiro, aumentamos o peso de um exame de 2 para 3. Depois, retiramos um atributo por vez. Dar peso zero a uma medida numérica retira sua diferença e seu peso da média. Para retirar o tabagismo, usamos apenas a parte numérica.

Por fim, mantemos as mesmas distâncias da referência e mudamos a votação. Os três vizinhos mais próximos valem dois votos cada e os outros oito valem um. Como são 14 votos ao todo, pode haver empate. Nesse caso, vale a classe do vizinho mais próximo.

In [23]:
def votar_com_peso(vizinhos, k):
    votos = Counter()

    for indice, (distancia, classe) in enumerate(vizinhos[:k]):
        peso_voto = 2 if indice < 3 else 1
        votos[classe] += peso_voto

    if votos["0"] == votos["1"]:
        return vizinhos[0][1]
    if votos["1"] > votos["0"]:
        return "1"
    return "0"


configuracoes_distancia = {
    "Referência": (pesos_submissao, ["SMOKER"]),
    "Glicose 3, hemoglobina 2": ({"GLUCOSE": 3, "HEMOGLOBIN": 2}, ["SMOKER"]),
    "Glicose 2, hemoglobina 3": ({"GLUCOSE": 2, "HEMOGLOBIN": 3}, ["SMOKER"]),
    "Sem SMOKER": (pesos_submissao, []),
}

for coluna in colunas_numericas:
    pesos_sem_coluna = pesos_submissao.copy()
    pesos_sem_coluna[coluna] = 0
    configuracoes_distancia[f"Sem {coluna}"] = (pesos_sem_coluna, ["SMOKER"])

resultados_variacoes = {nome: [] for nome in configuracoes_distancia}
resultados_variacoes["Voto duplo para os 3 primeiros"] = []
ignorar_glicose = ("GLUCOSE",)

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    referencia_rodada = montar_referencia(
        amostras_ajuste, parametros_rodada, categorias_rodada, ignorar_glicose
    )
    reais = [amostra["class"] for amostra in amostras_avaliacao]
    previstas_voto_duplo = []

    for nome, (pesos, categorias_usadas) in configuracoes_distancia.items():
        previstas = []
        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia_rodada, parametros_rodada, categorias_rodada,
                colunas=categorias_usadas, peso_categorico=0.10,
                pesos_numericos=pesos, ignorar_ausentes=ignorar_glicose
            )
            previstas.append(prever_classe(vizinhos, k_submissao))
            if nome == "Referência":
                previstas_voto_duplo.append(votar_com_peso(vizinhos, k_submissao))

        resultados_variacoes[nome].append(calcular_metricas(reais, previstas))

    resultados_variacoes["Voto duplo para os 3 primeiros"].append(
        calcular_metricas(reais, previstas_voto_duplo)
    )

#### Comparação dos resultados

A diferença de F1 está em pontos percentuais em relação à referência. A última coluna conta em quantas das cinco rodadas a alternativa teve F1 macro maior. Assim podemos distinguir um ganho que apareceu em várias rodadas de um ganho concentrado em poucas.

In [24]:
base_variacoes = resultados_variacoes["Referência"]
f1_base_variacoes = mean(rodada["f1_macro"] for rodada in base_variacoes)
medias_variacoes = {}

print("Versão                         | Acurácia | F1 macro | Dif. F1 (p.p.) | Recall 0 | Recall 1 | Rodadas melhores")
for nome, rodadas in resultados_variacoes.items():
    medias = {}
    for metrica in ["acuracia", "f1_macro", "recall_0", "recall_1"]:
        medias[metrica] = mean(rodada[metrica] for rodada in rodadas)
    medias_variacoes[nome] = medias

    diferenca = 100 * (medias["f1_macro"] - f1_base_variacoes)
    melhores = sum(
        rodada["f1_macro"] > base["f1_macro"]
        for rodada, base in zip(rodadas, base_variacoes)
    )
    placar = "-" if nome == "Referência" else f"{melhores}/5"

    print(
        f"{nome:30} | {medias['acuracia']:>8.2%} | {medias['f1_macro']:>8.2%} | "
        f"{diferenca:>+14.2f} | {medias['recall_0']:>8.2%} | "
        f"{medias['recall_1']:>8.2%} | {placar:>16}"
    )

Versão                         | Acurácia | F1 macro | Dif. F1 (p.p.) | Recall 0 | Recall 1 | Rodadas melhores
Referência                     |   88.69% |   88.68% |          +0.00 |   92.50% |   85.15% |                -
Glicose 3, hemoglobina 2       |   86.53% |   86.52% |          -2.16 |   92.00% |   81.46% |              1/5
Glicose 2, hemoglobina 3       |   88.94% |   88.92% |          +0.24 |   92.50% |   85.61% |              3/5
Sem SMOKER                     |   87.97% |   87.97% |          -0.71 |   92.00% |   84.24% |              1/5
Sem AGE                        |   87.02% |   87.00% |          -1.68 |   93.00% |   81.47% |              2/5
Sem HEIGHT                     |   86.77% |   86.75% |          -1.93 |   90.50% |   83.29% |              0/5
Sem WEIGHT                     |   87.01% |   87.00% |          -1.69 |   91.00% |   83.29% |              0/5
Sem SYSTOLIC                   |   85.81% |   85.79% |          -2.90 |   92.50% |   79.61% |              0/5
S

#### O que encontramos

O melhor resultado foi com peso 2 para glicose e 3 para hemoglobina. O F1 macro médio foi de 88,92%, um ganho de 0,24 ponto percentual sobre a referência. Essa versão melhorou em três das cinco rodadas e ficou com acurácia média de 88,94%.

Retirar qualquer um dos dez atributos reduziu o F1 médio. As maiores quedas foram ao retirar hemoglobina, glicose e triglicerídeos. Dar dois votos aos três vizinhos mais próximos também reduziu o resultado médio. Nesta comparação, mantemos todos os atributos e a votação simples como base para interpretar o ganho pequeno do novo peso.


### Quantos votos são necessários para prever diabetes?

Mantemos K igual a 11, peso 2 para glicose, peso 3 para hemoglobina e 10% para tabagismo quando comparável. Quando a glicose está ausente, ela fica fora da distância. As outras medidas ausentes continuam recebendo a mediana do treinamento.

Com a regra atual, são necessários pelo menos seis votos para prever diabetes. Testamos também uma regra que prevê diabetes a partir de cinco votos. Uma amostra com cinco votos para diabetes e seis para controle muda de classe com essa regra. Nos demais casos, a previsão fica igual.

Para saber o que ganhamos e perdemos, contamos quantas dessas mudanças corrigem um erro de diabetes e quantas criam um erro de controle. Usamos as mesmas cinco divisões e os mesmos vizinhos para as duas regras.

In [25]:
limites_votos = [6, 5]
resultados_limites = {limite: [] for limite in limites_votos}
corrigidos = 0
novos_erros = 0

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    referencia_rodada = montar_referencia(
        amostras_ajuste, parametros_rodada, categorias_rodada, ignorar_glicose
    )
    reais = []
    previstas_por_limite = {limite: [] for limite in limites_votos}

    for amostra in amostras_avaliacao:
        vizinhos = ordenar_vizinhos(
            amostra, referencia_rodada, parametros_rodada, categorias_rodada,
            colunas=["SMOKER"], peso_categorico=0.10,
            pesos_numericos={"GLUCOSE": 2, "HEMOGLOBIN": 3},
            ignorar_ausentes=ignorar_glicose
        )
        classe_real = amostra["class"]
        votos_diabetes = sum(classe == "1" for distancia, classe in vizinhos[:11])
        reais.append(classe_real)

        for limite in limites_votos:
            prevista = "1" if votos_diabetes >= limite else "0"
            previstas_por_limite[limite].append(prevista)

        if votos_diabetes == 5:
            if classe_real == "1":
                corrigidos += 1
            else:
                novos_erros += 1

    for limite, previstas in previstas_por_limite.items():
        resultados_limites[limite].append(calcular_metricas(reais, previstas))

In [26]:
print("Votos para diabetes | Acurácia | F1 macro | Recall 0 | Recall 1")
for limite, rodadas in resultados_limites.items():
    acuracia = mean(rodada["acuracia"] for rodada in rodadas)
    f1 = mean(rodada["f1_macro"] for rodada in rodadas)
    recall_0 = mean(rodada["recall_0"] for rodada in rodadas)
    recall_1 = mean(rodada["recall_1"] for rodada in rodadas)
    print(
        f"{limite:>19} | {acuracia:>8.2%} | {f1:>8.2%} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%}"
    )

rodadas_melhores = sum(
    r5["f1_macro"] > r6["f1_macro"]
    for r5, r6 in zip(resultados_limites[5], resultados_limites[6])
)
print()
print(f"Casos de diabetes corrigidos com 5 votos: {corrigidos}")
print(f"Novos erros na classe controle com 5 votos: {novos_erros}")
print(f"Rodadas com F1 maior com 5 votos: {rodadas_melhores}/5")

Votos para diabetes | Acurácia | F1 macro | Recall 0 | Recall 1
                  6 |   88.94% |   88.92% |   92.50% |   85.61%
                  5 |   88.69% |   88.66% |   87.50% |   89.78%

Casos de diabetes corrigidos com 5 votos: 9
Novos erros na classe controle com 5 votos: 10
Rodadas com F1 maior com 5 votos: 3/5


#### O que encontramos

Com cinco votos, corrigimos nove previsões de diabetes e passamos a errar dez amostras de controle. O recall de diabetes subiu de 85,61% para 89,78%, enquanto o recall de controle caiu de 92,50% para 87,50%.

O F1 macro foi maior em três rodadas com cinco votos, mas a média ficou um pouco abaixo da regra de seis. Como estamos comparando acurácia e F1 macro, mantemos seis votos como referência para os próximos testes.


### Manhattan ou Euclidiana nos atributos numéricos?

Com Manhattan, calculamos a média ponderada das diferenças absolutas. Com a Euclidiana, elevamos cada diferença ao quadrado, calculamos a média ponderada e tiramos a raiz. Por exemplo, duas diferenças de 0 e 2, com pesos iguais, dão 1 na Manhattan e aproximadamente 1,41 na Euclidiana. Assim, uma diferença grande em uma medida pode afastar mais um vizinho.

Dividimos pela soma dos pesos das medidas comparadas antes de tirar a raiz. Isso mantém a ideia de uma contribuição média por medida quando deixamos de comparar a glicose. Aplicamos D / (1 + D) às duas distâncias antes de combinar com tabagismo.

Mantemos K igual a 11, seis votos para prever diabetes, peso 2 para glicose e 3 para hemoglobina. Mantemos também o tabagismo com 10% da distância quando comparável e deixamos a glicose ausente fora da comparação. Calculamos medianas, quartis e categorias com as amostras de referência de cada rodada.

In [27]:
metricas_distancia = ["manhattan", "euclidiana"]
resultados_distancias = {metrica: [] for metrica in metricas_distancia}
pesos_comparacao = {"GLUCOSE": 2, "HEMOGLOBIN": 3}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    referencia_rodada = montar_referencia(
        amostras_ajuste, parametros_rodada, categorias_rodada, ignorar_glicose
    )
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for metrica in metricas_distancia:
        previstas = []
        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia_rodada, parametros_rodada, categorias_rodada,
                colunas=["SMOKER"], peso_categorico=0.10,
                pesos_numericos=pesos_comparacao, ignorar_ausentes=ignorar_glicose,
                metrica_numerica=metrica
            )
            previstas.append(prever_classe(vizinhos, 11))

        resultados_distancias[metrica].append(calcular_metricas(reais, previstas))

In [28]:
print("Distância  | Acurácia | F1 macro | Recall 0 | Recall 1")
for metrica, rodadas in resultados_distancias.items():
    acuracia = mean(rodada["acuracia"] for rodada in rodadas)
    f1 = mean(rodada["f1_macro"] for rodada in rodadas)
    recall_0 = mean(rodada["recall_0"] for rodada in rodadas)
    recall_1 = mean(rodada["recall_1"] for rodada in rodadas)
    print(
        f"{metrica:10} | {acuracia:>8.2%} | {f1:>8.2%} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%}"
    )

print()
print("Rodada | F1 Manhattan | F1 Euclidiana")
for numero, (manhattan, euclidiana) in enumerate(
    zip(resultados_distancias["manhattan"], resultados_distancias["euclidiana"]),
    start=1
):
    print(f"{numero:>6} | {manhattan['f1_macro']:>12.2%} | {euclidiana['f1_macro']:>13.2%}")

Distância  | Acurácia | F1 macro | Recall 0 | Recall 1
manhattan  |   88.94% |   88.92% |   92.50% |   85.61%
euclidiana |   85.32% |   85.30% |   89.50% |   81.43%

Rodada | F1 Manhattan | F1 Euclidiana
     1 |       91.64% |        91.64%
     2 |       90.36% |        83.13%
     3 |       84.34% |        81.89%
     4 |       89.13% |        85.54%
     5 |       89.16% |        84.33%


#### O que encontramos

Manhattan teve F1 macro maior em quatro rodadas e empatou na primeira. A acurácia média ficou em 88,94% com Manhattan e 85,32% com Euclidiana. Os recalls de controle e diabetes também foram maiores com Manhattan.

A Euclidiana dá mais influência a diferenças grandes em uma medida. Isso muda a ordem dos vizinhos. Nesta comparação, os vizinhos escolhidos com Manhattan deram melhores resultados, então mantemos essa distância na configuração de referência.


### Quanto devem pesar os triglicerídeos?

Em um teste anterior, retirar os triglicerídeos reduziu o F1 macro médio. Agora comparamos peso 1 e peso 2 para essa medida. Com peso 2, uma diferença ajustada de 0,4 nos triglicerídeos contribui com 0,8 para a soma da distância numérica. Dividimos pela soma dos pesos como antes.

Mantemos Manhattan, K igual a 11, votação simples, peso 2 para glicose e 3 para hemoglobina. O tabagismo e o tratamento dos valores ausentes seguem as mesmas regras. Usamos as mesmas cinco divisões e ajustamos os parâmetros com as quatro partes de referência em cada rodada.

In [29]:
pesos_triglicerideos = [1, 2]
resultados_triglicerideos = {peso: [] for peso in pesos_triglicerideos}

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    referencia_rodada = montar_referencia(
        amostras_ajuste, parametros_rodada, categorias_rodada, ignorar_glicose
    )
    reais = [amostra["class"] for amostra in amostras_avaliacao]

    for peso in pesos_triglicerideos:
        pesos = {"GLUCOSE": 2, "HEMOGLOBIN": 3, "TRIGLYCERIDES": peso}
        previstas = []
        for amostra in amostras_avaliacao:
            vizinhos = ordenar_vizinhos(
                amostra, referencia_rodada, parametros_rodada, categorias_rodada,
                colunas=["SMOKER"], peso_categorico=0.10,
                pesos_numericos=pesos, ignorar_ausentes=ignorar_glicose
            )
            previstas.append(prever_classe(vizinhos, 11))

        resultados_triglicerideos[peso].append(calcular_metricas(reais, previstas))

In [30]:
print("Peso dos triglicerídeos | Acurácia | F1 macro | Recall 0 | Recall 1")
for peso, rodadas in resultados_triglicerideos.items():
    acuracia = mean(rodada["acuracia"] for rodada in rodadas)
    f1 = mean(rodada["f1_macro"] for rodada in rodadas)
    recall_0 = mean(rodada["recall_0"] for rodada in rodadas)
    recall_1 = mean(rodada["recall_1"] for rodada in rodadas)

    print(
        f"{peso:>23} | {acuracia:>8.2%} | {f1:>8.2%} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%}"
    )

print()
print("Rodada | F1 com peso 1 | F1 com peso 2")
for numero, (peso_1, peso_2) in enumerate(
    zip(resultados_triglicerideos[1], resultados_triglicerideos[2]), start=1
):
    print(f"{numero:>6} | {peso_1['f1_macro']:>13.2%} | {peso_2['f1_macro']:>13.2%}")

Peso dos triglicerídeos | Acurácia | F1 macro | Recall 0 | Recall 1
                      1 |   88.94% |   88.92% |   92.50% |   85.61%
                      2 |   88.21% |   88.20% |   92.50% |   84.23%

Rodada | F1 com peso 1 | F1 com peso 2
     1 |        91.64% |        91.66%
     2 |        90.36% |        90.36%
     3 |        84.34% |        81.92%
     4 |        89.13% |        89.13%
     5 |        89.16% |        87.95%


#### O que encontramos

Dobrar o peso dos triglicerídeos reduziu a acurácia média de 88,94% para 88,21% e o F1 macro de 88,92% para 88,20%. O recall de controle ficou igual, enquanto o de diabetes caiu de 85,61% para 84,23%.

Com peso 2, o F1 melhorou ligeiramente em uma rodada, empatou em duas e piorou em duas. Mantemos os triglicerídeos no modelo com peso 1.


### Quartis ou desvio padrão para ajustar a escala?

Na regra atual, subtraímos a mediana e dividimos pela diferença entre o terceiro e o primeiro quartil. Na outra regra, subtraímos a média e dividimos pelo desvio padrão amostral. Para a distância entre duas medidas, subtrair o mesmo centro das duas não muda a diferença. O que muda a distância é o divisor de cada coluna.

Nas duas versões, continuamos preenchendo os valores ausentes pela mediana, com exceção da glicose. Quando a glicose falta em uma das amostras, ela fica fora da distância. Calculamos os parâmetros só com os valores disponíveis nas quatro partes de referência de cada rodada.

Mantemos Manhattan, K igual a 11 e votação simples. A glicose tem peso 2, a hemoglobina tem peso 3 e o tabagismo continua com 10% da distância quando comparável. Escolhemos a escala pelo maior F1 macro médio nas cinco rodadas. Em caso de empate exato, ficamos com os quartis.

In [31]:
escalas = {"quartis": "Mediana e quartis", "desvio": "Média e desvio padrão"}
resultados_escalas = {escala: [] for escala in escalas}


def prever_candidata_escala(amostras_ajuste, amostras_avaliacao, escala):
    parametros = calcular_parametros_numericos(amostras_ajuste, escala)
    categorias = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}
    referencia = montar_referencia(
        amostras_ajuste, parametros, categorias, ignorar_glicose
    )
    previstas = []

    for amostra in amostras_avaliacao:
        vizinhos = ordenar_vizinhos(
            amostra, referencia, parametros, categorias,
            colunas=["SMOKER"], peso_categorico=0.10,
            pesos_numericos=pesos_comparacao, ignorar_ausentes=ignorar_glicose
        )
        previstas.append(prever_classe(vizinhos, 11))

    return previstas


for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    reais = [amostra["class"] for amostra in amostras_avaliacao]
    for escala in escalas:
        previstas = prever_candidata_escala(amostras_ajuste, amostras_avaliacao, escala)
        resultados_escalas[escala].append(calcular_metricas(reais, previstas))

In [32]:
medias_escalas = {}

print("Escala                 | Acurácia | F1 macro | Recall 0 | Recall 1")
for escala, rodadas in resultados_escalas.items():
    medias = {}
    for metrica in ["acuracia", "f1_macro", "recall_0", "recall_1"]:
        medias[metrica] = mean(rodada[metrica] for rodada in rodadas)
    medias_escalas[escala] = medias
    print(
        f"{escalas[escala]:22} | {medias['acuracia']:>8.2%} | "
        f"{medias['f1_macro']:>8.2%} | {medias['recall_0']:>8.2%} | "
        f"{medias['recall_1']:>8.2%}"
    )

print()
print("Rodada | F1 com quartis | F1 com desvio padrão")
for numero, (quartis, desvio) in enumerate(
    zip(resultados_escalas["quartis"], resultados_escalas["desvio"]), start=1
):
    print(f"{numero:>6} | {quartis['f1_macro']:>14.2%} | {desvio['f1_macro']:>20.2%}")

melhor_escala = max(escalas, key=lambda escala: medias_escalas[escala]["f1_macro"])
print()
print(f"Escala escolhida: {escalas[melhor_escala]}")

Escala                 | Acurácia | F1 macro | Recall 0 | Recall 1
Mediana e quartis      |   88.94% |   88.92% |   92.50% |   85.61%
Média e desvio padrão  |   83.63% |   83.62% |   87.00% |   80.51%

Rodada | F1 com quartis | F1 com desvio padrão
     1 |         91.64% |               91.66%
     2 |         90.36% |               84.34%
     3 |         84.34% |               79.51%
     4 |         89.13% |               81.89%
     5 |         89.16% |               80.72%

Escala escolhida: Mediana e quartis


#### Comparação nas 104 amostras reservadas

Fixamos a candidata com a escala escolhida nas cinco rodadas. Usamos as 416 amostras de treinamento como referência e comparamos as previsões com as da configuração já enviada ao Kaggle, nas mesmas 104 amostras. Contamos também quantos erros a candidata corrige e quantos acertos anteriores ela perde.

Essas amostras já foram consultadas antes. Esta comparação serve como referência local para decidir se vale um novo envio.

In [33]:
previsoes_candidata = prever_candidata_escala(treinamento, validacao, melhor_escala)
metricas_candidata = calcular_metricas(reais_validacao, previsoes_candidata)

print("Versão             | Acertos | Acurácia | F1 macro | Recall 0 | Recall 1")
for nome, metricas in [
    ("Enviada ao Kaggle", metricas_reservadas),
    ("Candidata", metricas_candidata),
]:
    matriz = metricas["matriz"]
    acertos = matriz[0][0] + matriz[1][1]
    print(
        f"{nome:18} | {acertos:>7} | {metricas['acuracia']:>8.2%} | "
        f"{metricas['f1_macro']:>8.2%} | {metricas['recall_0']:>8.2%} | "
        f"{metricas['recall_1']:>8.2%}"
    )

erros_corrigidos = 0
acertos_perdidos = 0
for real, enviada, candidata in zip(
    reais_validacao, previsoes_reservadas, previsoes_candidata
):
    if enviada != real and candidata == real:
        erros_corrigidos += 1
    if enviada == real and candidata != real:
        acertos_perdidos += 1

print()
print(f"Erros corrigidos pela candidata: {erros_corrigidos}")
print(f"Acertos da versão enviada que a candidata perdeu: {acertos_perdidos}")

Versão             | Acertos | Acurácia | F1 macro | Recall 0 | Recall 1
Enviada ao Kaggle  |      90 |   86.54% |   86.53% |   88.00% |   85.19%
Candidata          |      88 |   84.62% |   84.62% |   88.00% |   81.48%

Erros corrigidos pela candidata: 3
Acertos da versão enviada que a candidata perdeu: 5


#### O que encontramos

Mediana e quartis tiveram F1 macro maior em quatro das cinco rodadas. Com média e desvio padrão, a acurácia média caiu de 88,94% para 83,63%.

Nas 104 amostras reservadas, a candidata corrigiu três erros, mas perdeu cinco acertos da versão enviada. O recall de controle ficou em 88,00% nas duas. O recall de diabetes caiu de 85,19% para 81,48%. O ganho que vimos nas cinco rodadas ao ajustar a candidata não apareceu nesta avaliação. Por isso, mantemos a versão enviada como referência para o Kaggle.


### O que muda com o peso da hemoglobina e a glicose ausente?

A versão enviada ao Kaggle dá peso 2 para os dois exames e preenche a glicose ausente pela mediana. A versão que ignora a glicose ausente e mantém peso 2 para os dois exames chegou a 88,69% de acurácia média nas cinco rodadas. Agora avaliamos essa versão nas 104 amostras reservadas.

Comparamos quatro configurações. Uma muda só o peso da hemoglobina, outra muda só a regra da glicose, e a última combina as duas mudanças. Mantemos as mesmas 416 amostras de referência, Manhattan, mediana e quartis, K igual a 11 e votação simples.

Na tabela, os erros corrigidos e os novos erros são contados em relação à versão enviada. Como já consultamos as 104 amostras em outros testes, usamos a comparação para entender o efeito de cada mudança.

In [34]:
versoes_reservadas = {
    "Já enviada": (2, ()),
    "Só hemoglobina com peso 3": (3, ()),
    "Só ignorar glicose ausente": (2, ("GLUCOSE",)),
    "As duas mudanças": (3, ("GLUCOSE",)),
}
previsoes_por_versao = {}

for nome, (peso_hemoglobina, ignorar) in versoes_reservadas.items():
    referencia = montar_referencia(
        treinamento, parametros_numericos, categorias_por_coluna, ignorar
    )
    pesos = {"GLUCOSE": 2, "HEMOGLOBIN": peso_hemoglobina}
    previstas = []

    for amostra in validacao:
        vizinhos = ordenar_vizinhos(
            amostra, referencia, parametros_numericos, categorias_por_coluna,
            colunas=["SMOKER"], peso_categorico=0.10,
            pesos_numericos=pesos, ignorar_ausentes=ignorar
        )
        previstas.append(prever_classe(vizinhos, 11))

    previsoes_por_versao[nome] = previstas

In [35]:
metricas_por_versao = {}
mudancas_por_versao = {}

print("Versão                     | Acertos | Acurácia | F1 macro | Recall 0 | Recall 1 | Erros corrigidos | Novos erros")
for nome, previstas in previsoes_por_versao.items():
    metricas = calcular_metricas(reais_validacao, previstas)
    metricas_por_versao[nome] = metricas

    matriz = metricas["matriz"]
    acertos = matriz[0][0] + matriz[1][1]
    corrigidos = 0
    novos_erros = 0
    for real, enviada, prevista in zip(reais_validacao, previsoes_reservadas, previstas):
        if enviada != real and prevista == real:
            corrigidos += 1
        if enviada == real and prevista != real:
            novos_erros += 1
    mudancas_por_versao[nome] = (corrigidos, novos_erros)

    print(
        f"{nome:26} | {acertos:>7} | {metricas['acuracia']:>8.2%} | "
        f"{metricas['f1_macro']:>8.2%} | {metricas['recall_0']:>8.2%} | "
        f"{metricas['recall_1']:>8.2%} | {corrigidos:>16} | {novos_erros:>11}"
    )

Versão                     | Acertos | Acurácia | F1 macro | Recall 0 | Recall 1 | Erros corrigidos | Novos erros
Já enviada                 |      90 |   86.54% |   86.53% |   88.00% |   85.19% |                0 |           0
Só hemoglobina com peso 3  |      89 |   85.58% |   85.58% |   88.00% |   83.33% |                1 |           2
Só ignorar glicose ausente |      88 |   84.62% |   84.62% |   88.00% |   81.48% |                3 |           5
As duas mudanças           |      88 |   84.62% |   84.62% |   88.00% |   81.48% |                3 |           5


#### O que encontramos

Nas 104 amostras reservadas, a versão enviada acertou 90 casos. Mudar só o peso da hemoglobina para 3 corrigiu um erro e criou dois, com saldo de um acerto a menos. Ignorar só a glicose ausente, a versão que alcançou 88,69% nas cinco rodadas, corrigiu três erros e criou cinco, com saldo de dois acertos a menos.

As duas mudanças juntas também resultaram em 88 acertos. Nesta comparação local, a versão enviada teve o maior F1 macro e encontrou mais casos de diabetes. A versão de 88,69% foi avaliada localmente, enquanto o resultado público disponível no Kaggle continua sendo o da versão enviada.


### Procurar vizinhos com os mesmos exames disponíveis

Se a amostra tem glicose medida, a versão candidata procura vizinhos que também tenham glicose medida. A mesma regra vale para a hemoglobina. Se a amostra tem os dois exames, exigimos os dois nos vizinhos. Se faltam os dois, qualquer amostra pode ser vizinha.

A regra muda quem pode entrar na votação. Mantemos o preenchimento pela mediana, os pesos e a distância da versão enviada ao Kaggle. Se o filtro deixar menos de 11 candidatos, usamos a referência completa naquela previsão. Assim, conseguimos testar o filtro sem mudar também o cálculo da distância.

Usamos as mesmas cinco divisões e comparamos os erros por quantidade de exames disponíveis. Isso permite ver se o filtro ajuda mais quando temos dois exames ou só um.

In [36]:
exames_filtro = ("GLUCOSE", "HEMOGLOBIN")
perfis_exames = [(), ("GLUCOSE",), ("HEMOGLOBIN",), exames_filtro]


def exames_disponiveis(amostra):
    return tuple(exame for exame in exames_filtro if ler_numero(amostra, exame) is not None)


resultados_filtro = {"Versão enviada": [], "Com filtro de exames": []}
erros_filtro_por_grupo = {
    quantidade: {"total": 0, "erros_sem_filtro": 0, "erros_com_filtro": 0}
    for quantidade in range(3)
}
vezes_sem_vizinhos = 0

for indice, amostras_avaliacao in enumerate(partes):
    amostras_ajuste = []
    for outro_indice, parte in enumerate(partes):
        if outro_indice != indice:
            amostras_ajuste.extend(parte)

    parametros_rodada = calcular_parametros_numericos(amostras_ajuste)
    categorias_rodada = {"SMOKER": identificar_categorias(amostras_ajuste)["SMOKER"]}

    referencia_com_perfil = []
    for amostra in amostras_ajuste:
        linha_referencia = (
            preparar_numericos(amostra, parametros_rodada),
            codificar_categorias(amostra, categorias_rodada),
            amostra["class"],
        )
        referencia_com_perfil.append((linha_referencia, exames_disponiveis(amostra)))

    random.Random(42).shuffle(referencia_com_perfil)
    referencias_por_perfil = {}
    for perfil in perfis_exames:
        referencias_por_perfil[perfil] = [
            linha for linha, disponiveis in referencia_com_perfil
            if all(exame in disponiveis for exame in perfil)
        ]

    reais = []
    previstas_sem_filtro = []
    previstas_com_filtro = []

    for amostra in amostras_avaliacao:
        perfil = exames_disponiveis(amostra)
        referencia_filtrada = referencias_por_perfil[perfil]
        if len(referencia_filtrada) < k_submissao:
            referencia_filtrada = referencias_por_perfil[()]
            vezes_sem_vizinhos += 1

        vizinhos_sem_filtro = ordenar_vizinhos(
            amostra, referencias_por_perfil[()], parametros_rodada, categorias_rodada,
            colunas=["SMOKER"], peso_categorico=0.10, pesos_numericos=pesos_submissao
        )
        vizinhos_com_filtro = ordenar_vizinhos(
            amostra, referencia_filtrada, parametros_rodada, categorias_rodada,
            colunas=["SMOKER"], peso_categorico=0.10, pesos_numericos=pesos_submissao
        )
        sem_filtro = prever_classe(vizinhos_sem_filtro, k_submissao)
        com_filtro = prever_classe(vizinhos_com_filtro, k_submissao)
        real = amostra["class"]

        reais.append(real)
        previstas_sem_filtro.append(sem_filtro)
        previstas_com_filtro.append(com_filtro)

        grupo = erros_filtro_por_grupo[len(perfil)]
        grupo["total"] += 1
        grupo["erros_sem_filtro"] += sem_filtro != real
        grupo["erros_com_filtro"] += com_filtro != real

    resultados_filtro["Versão enviada"].append(calcular_metricas(reais, previstas_sem_filtro))
    resultados_filtro["Com filtro de exames"].append(calcular_metricas(reais, previstas_com_filtro))

In [37]:
print("Versão               | Acurácia | F1 macro | Recall 0 | Recall 1")
for nome, rodadas in resultados_filtro.items():
    acuracia = mean(rodada["acuracia"] for rodada in rodadas)
    f1 = mean(rodada["f1_macro"] for rodada in rodadas)
    recall_0 = mean(rodada["recall_0"] for rodada in rodadas)
    recall_1 = mean(rodada["recall_1"] for rodada in rodadas)
    print(
        f"{nome:20} | {acuracia:>8.2%} | {f1:>8.2%} | "
        f"{recall_0:>8.2%} | {recall_1:>8.2%}"
    )

melhoras_filtro = sum(
    com["f1_macro"] > sem["f1_macro"]
    for com, sem in zip(resultados_filtro["Com filtro de exames"], resultados_filtro["Versão enviada"])
)

print()
print("Exames disponíveis | Amostras | Erros sem filtro | Erros com filtro")
for quantidade, grupo in erros_filtro_por_grupo.items():
    print(
        f"{quantidade:>18} | {grupo['total']:>8} | "
        f"{grupo['erros_sem_filtro']:>16} | {grupo['erros_com_filtro']:>16}"
    )
print()
print(f"Rodadas com F1 maior após filtrar: {melhoras_filtro}/5")
print(f"Previsões que precisaram voltar à referência completa: {vezes_sem_vizinhos}")

Versão               | Acurácia | F1 macro | Recall 0 | Recall 1
Versão enviada       |   88.21% |   88.18% |   93.50% |   83.28%
Com filtro de exames |   85.57% |   85.52% |   95.00% |   76.84%

Exames disponíveis | Amostras | Erros sem filtro | Erros com filtro
                 0 |       13 |                3 |                3
                 1 |      121 |               20 |               20
                 2 |      282 |               26 |               37

Rodadas com F1 maior após filtrar: 0/5
Previsões que precisaram voltar à referência completa: 0


#### O que encontramos

O filtro reduziu a acurácia média de 88,21% para 85,57% e o F1 macro de 88,18% para 85,52%. O recall de controle subiu de 93,50% para 95,00%, enquanto o de diabetes caiu de 83,28% para 76,84%.

Os 11 erros a mais se concentraram nas amostras com os dois exames disponíveis, justamente o grupo em que o filtro exigiu mais dos vizinhos. Nos grupos com zero ou um exame, o número de erros permaneceu igual. Como o F1 não melhorou em nenhuma rodada, mantemos a referência completa para procurar os vizinhos.


### Segunda submissão ao Kaggle

Nesta versão, mantemos Manhattan, K igual a 11, votação simples, os pesos 2 para glicose e hemoglobina e o tabagismo. A única mudança é deixar a glicose fora da distância quando ela falta em uma das amostras. Essa versão teve 88,69% de acurácia média nas cinco rodadas e 84,62% nas 104 amostras reservadas.

Usamos novamente as 520 amostras rotuladas como referência e mantemos a ordem dos IDs do teste. Salvamos as previsões em um arquivo separado para comparar as duas submissões.

In [ ]:
glicose_fora_quando_ausente = ("GLUCOSE",)
referencia_glicose = montar_referencia(
    amostras, parametros_submissao, categorias_submissao, glicose_fora_quando_ausente
)
previsoes_glicose = []

for amostra in entradas_teste:
    vizinhos = ordenar_vizinhos(
        amostra, referencia_glicose, parametros_submissao, categorias_submissao,
        colunas=["SMOKER"], peso_categorico=0.10, pesos_numericos=pesos_submissao,
        ignorar_ausentes=glicose_fora_quando_ausente
    )
    previsoes_glicose.append(prever_classe(vizinhos, k_submissao))

caminho_submissao_glicose = (
    caminho.parent.parent / "resultados" / "submissao_knn_k11_glicose_ausente.csv"
)
linhas_glicose = list(zip(ids_teste, previsoes_glicose, strict=True))
if any(classe not in {"0", "1"} for _, classe in linhas_glicose):
    raise ValueError("As previsões devem pertencer às classes 0 ou 1.")

with caminho_submissao_glicose.open("w", encoding="utf-8", newline="") as arquivo_submissao:
    escritor = csv.writer(arquivo_submissao)
    escritor.writerow(["id", "class"])
    escritor.writerows(linhas_glicose)

print(f"Arquivo salvo: resultados/{caminho_submissao_glicose.name}")
print(f"Linhas de previsão: {len(linhas_glicose)}")

Arquivo salvo: resultados/submissao_knn_k11_glicose_ausente.csv
Linhas de previsão: 312


#### Resultado das duas submissões

| Versão | Acurácia média nas cinco rodadas | Acurácia nas 104 amostras reservadas | Pontuação pública no Kaggle |
|---|---:|---:|---:|
| Primeira submissão, glicose ausente preenchida pela mediana | 88,21% | 86,54% | 0,82897 |
| Segunda submissão, glicose ausente fora da distância | 88,69% | 84,62% | 0,80570 |

A segunda regra mudou 14 das 312 previsões do teste. A pontuação pública caiu 0,02327. O pequeno ganho de acurácia nas cinco rodadas não apareceu nas 104 amostras reservadas nem no placar público. Mantemos a primeira versão como referência para a comparação com os próximos modelos.
